# Chatbot/Agent with Memory

In [ ]:
# Tools for describing the shape of our data (see the State class below)
from typing import Annotated, TypedDict
# StateGraph builds the bot's flowchart; START and END mark where the flow begins and stops
from langgraph.graph import StateGraph, START, END
# add_messages makes new messages get ADDED to the chat history instead of replacing it
from langgraph.graph.message import add_messages

In [ ]:
# The State is the bot's notepad: everything it keeps track of while it runs
class State(TypedDict):
    # A list holding the whole conversation; add_messages keeps adding new messages to the end
    messages : Annotated[list, add_messages]

In [ ]:
# os lets Python read settings from your computer, like your API keys
import os
# load_dotenv can read the secret keys stored in your .env file
from dotenv import load_dotenv
# Load the keys from .env so the code can use them (shows True if it found the file)
load_dotenv()

In [ ]:
# A helper that can create chat models from many AI companies (imported, but not used below)
from langchain.chat_models import init_chat_model
# The connector that lets us talk to Google's Gemini AI
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
# Create the AI model: the "brain" that reads messages and writes replies
llm = ChatGoogleGenerativeAI(
    # Which Gemini model to use
    model = "gemini-flash-lite-latest",
    # 0 means no creativity/randomness: the same question gives (almost) the same answer
    temperature = 0.0
)

In [ ]:
# The chatbot step: hand the conversation so far to the AI
def chatbot(state:State):
    # Ask the AI for a reply and add that reply to the conversation
    return {"messages": [llm.invoke(state["messages"])]}

In [ ]:
# Bring in the LangChain library itself
import langchain
# Show which version of LangChain is installed
print(langchain.__version__)

In [ ]:
# TavilySearch is a ready-made tool that lets the AI search the internet
from langchain_tavily import TavilySearch

# Create the search tool; it brings back only the top 1 result
tool = TavilySearch(max_results = 1)
# Try the search tool once by hand to see what it returns
tool.invoke("How do i eat out of my elbow")

In [ ]:
# Our own tiny tool: multiply two whole numbers.
# The text in triple quotes below is what the AI reads to know when to use it
def multiply(a: int, b: int) -> int:
    """Multiply a and b

    Args:
        a (int) :  first int
        b (int) :  second int

    Returns:
        int : output integer
    """
    # Give back a times b
    return a * b

In [ ]:
# Put both tools (web search and multiply) in a list
tools = [tool, multiply]

In [ ]:
# Tell the AI about our tools so it can choose to use them
llm_with_tools = llm.bind_tools(tools)

In [ ]:
# ToolNode is a ready-made box that actually runs whatever tool the AI asks for
from langgraph.prebuilt import ToolNode
# tools_condition looks at the AI's reply: did it ask for a tool, or is it finished?
from langgraph.prebuilt import tools_condition

In [ ]:
# The main step: hand the conversation to the AI that knows about our tools
def tool_calling_llm(state:State):
    # The AI either answers or asks to use a tool; add its reply to the conversation
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

## Initializing an Instance of the Memory Saver Class

Kind of like RAM for agent context

In [ ]:
# MemorySaver keeps past conversations in the computer's memory (lost when you restart)
from langgraph.checkpoint.memory import MemorySaver
# Create an empty memory store
memory = MemorySaver()

In [ ]:
# Start an empty flowchart that uses our State as its notepad
graph_builder = StateGraph(State)

# Add the box where the AI thinks and replies
graph_builder.add_node("tool_calling_llm", tool_calling_llm)
# Add the box that runs the tools
graph_builder.add_node("tools", ToolNode(tools=tools))

# Arrow: when the bot starts, go to the AI first
graph_builder.add_edge(START, "tool_calling_llm")
#condtional edges
# A fork in the road after the AI box (which way depends on its reply)
graph_builder.add_conditional_edges(
    # The fork starts at the AI box...
    "tool_calling_llm", 
    # ...go to "tools" if the AI asked for a tool, otherwise stop
    tools_condition
    )
# Arrow: after a tool runs, go back to the AI so it can read the result.
# This loop (think -> act -> observe -> think again) is what makes it a ReAct agent
graph_builder.add_edge("tools", "tool_calling_llm")

# Finish the flowchart and connect the memory, so the bot remembers earlier messages
graph = graph_builder.compile(checkpointer = memory)

In [ ]:
# Show a picture of the flowchart
graph

## Configuring the Memory for agent

In [ ]:
# A "thread" is one conversation. Using the same thread_id means the bot remembers this chat
config = {"configurable" : {"thread_id" : 1}}

## Testing the Graph for memory

In [ ]:
# Tell the bot some facts, inside conversation thread 1
response = graph.invoke({"messages": "My name is Aryan and my roomate is Raghav."}, config = config)
# Go through every message in the conversation...
for m in response['messages']:
    # ...and print it in a neat, readable format
    m.pretty_print()

In [ ]:
# Same thread, so the bot should remember what we just told it
response = graph.invoke({"messages": "What all do you  know about me?"}, config = config)
# Go through every message in the conversation...
for m in response['messages']:
    # ...and print it in a neat, readable format
    m.pretty_print()

In [ ]:
# Show just the text of the last message (the bot's final answer)
response['messages'][-1].content